In [1]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"faziaakhtardcs","key":"2d7483f17b58e2a03f7d163838920f9e"}'}

In [2]:
import os
os.makedirs('/root/.kaggle', exist_ok=True)
os.rename('kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 600)

In [3]:
!kaggle datasets download -d vanshikavmittal/fakeddit-dataset
!unzip -o fakeddit-dataset.zip -d data

Dataset URL: https://www.kaggle.com/datasets/vanshikavmittal/fakeddit-dataset
License(s): unknown
100% 72.8M/72.8M [00:02<00:00, 29.4MB/s]

Archive:  fakeddit-dataset.zip
  inflating: data/multimodal_only_samples/multimodal_test_public.tsv  
  inflating: data/multimodal_only_samples/multimodal_train.tsv  
  inflating: data/multimodal_only_samples/multimodal_validate.tsv  


In [4]:
import pandas as pd

train_df = pd.read_csv('data/multimodal_only_samples/multimodal_train.tsv', sep='\t')
print(train_df.shape)
train_df.head()

(564000, 16)


,author,clean_title,created_utc,domain,hasImage,id,image_url,linked_submission_id,num_comments,score,subreddit,title,upvote_ratio,2_way_label,3_way_label,6_way_label
0,Alexithymia,my walgreens offbrand mucinex was engraved wit...,1.551641e+09,i.imgur.com,True,awxhir,https://external-preview.redd.it/WylDbZrnbvZdB...,NaN,2.0,12,mildlyinteresting,My Walgreens offbrand Mucinex was engraved wit...,0.84,1,0,0
1,VIDCAs17,this concerned sink with a tiny hat,1.534727e+09,i.redd.it,True,98pbid,https://preview.redd.it/wsfx0gp0f5h11.jpg?widt...,NaN,2.0,119,pareidolia,This concerned sink with a tiny hat,0.99,0,2,2
2,prometheus1123,hackers leak emails from uae ambassador to us,1.496511e+09,aljazeera.com,True,6f2cy5,https://external-preview.redd.it/6fNhdbc6K1vFA...,NaN,1.0,44,neutralnews,Hackers leak emails from UAE ambassador to US,0.92,1,0,0
3,NaN,puppy taking in the view,1.471341e+09,i.imgur.com,True,4xypkv,https://external-preview.redd.it/HLtVNhTR6wtYt...,NaN,26.0,250,photoshopbattles,PsBattle: Puppy taking in the view,0.95,1,0,0
4,3rikR3ith,i found a face in my sheet music too,1.525318e+09,i.redd.it,True,8gnet9,https://preview.redd.it/ri7ut2wn8kv01.jpg?widt...,NaN,2.0,13,pareidolia,I found a face in my sheet music too!,0.84,0,2,2


In [5]:
# Keep only rows that actually have images
df = train_df[train_df['hasImage'] == True].copy()

# Take a manageable random sample
df_sample = df.sample(2000, random_state=42).reset_index(drop=True)

df_sample = df_sample[['clean_title', 'image_url', '2_way_label']]
df_sample.columns = ['text', 'image_url', 'label']

print(df_sample.shape)
print(df_sample['label'].value_counts())
df_sample.head()

(2000, 3)
label
0    1203
1     797
Name: count, dtype: int64


,text,image_url,label
0,feeling lucky,http://i.imgur.com/Zu6Sx.jpg,0
1,cutouts,https://31.media.tumblr.com/d7100866f676a6a376...,0
2,my ceiling looks like an sd card,https://preview.redd.it/zqg8c8fteqe31.jpg?widt...,0
3,join the raaf,https://external-preview.redd.it/q9DNAI6S1OC2v...,0
4,hangover,http://i.imgur.com/wLCYVSv.jpg,0


In [6]:
import requests
import os
from PIL import Image
from io import BytesIO

os.makedirs('images', exist_ok=True)

def download_image(url, save_path, timeout=5):
    try:
        response = requests.get(url, timeout=timeout, headers={'User-Agent': 'Mozilla/5.0'})
        img = Image.open(BytesIO(response.content)).convert('RGB')
        img.save(save_path)
        return True
    except Exception:
        return False

success_flags = []
for idx, row in df_sample.iterrows():
    save_path = f"images/{idx}.jpg"
    ok = download_image(row['image_url'], save_path)
    success_flags.append(ok)
    if idx % 200 == 0:
        print(f"Processed {idx}/{len(df_sample)}")

df_sample['image_downloaded'] = success_flags
print(df_sample['image_downloaded'].value_counts())

Processed 0/2000
Processed 200/2000
Processed 400/2000
Processed 600/2000


/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Processed 800/2000
Processed 1000/2000
Processed 1200/2000
Processed 1400/2000
Processed 1600/2000
Processed 1800/2000
image_downloaded
True     1895
False     105
Name: count, dtype: int64


In [7]:
df_final = df_sample[df_sample['image_downloaded'] == True].reset_index(drop=True)
df_final['image_path'] = df_final.index.map(lambda i: f"images/{df_sample[df_sample['image_downloaded']==True].index[i]}.jpg")
print(df_final.shape)

(1895, 5)


In [8]:
df_final = df_sample[df_sample['image_downloaded'] == True].copy()
df_final['image_path'] = df_final.index.map(lambda i: f"images/{i}.jpg")
df_final = df_final.reset_index(drop=True)
print(df_final.shape)
df_final.head()

(1895, 5)


,text,image_url,label,image_downloaded,image_path
0,feeling lucky,http://i.imgur.com/Zu6Sx.jpg,0,True,images/0.jpg
1,cutouts,https://31.media.tumblr.com/d7100866f676a6a376...,0,True,images/1.jpg
2,my ceiling looks like an sd card,https://preview.redd.it/zqg8c8fteqe31.jpg?widt...,0,True,images/2.jpg
3,join the raaf,https://external-preview.redd.it/q9DNAI6S1OC2v...,0,True,images/3.jpg
4,hangover,http://i.imgur.com/wLCYVSv.jpg,0,True,images/4.jpg


In [9]:
!pip install -q git+https://github.com/openai/CLIP.git

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.9 MB/s eta 0:00:00


In [10]:
import torch
import clip

device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = clip.load("ViT-B/32", device=device)
print("CLIP loaded on:", device)

100%|███████████████████████████████████████| 338M/338M [00:03<00:00, 88.7MiB/s]


CLIP loaded on: cuda


In [11]:
import torch
from PIL import Image
from tqdm import tqdm

image_features_list = []
text_features_list = []

for idx, row in tqdm(df_final.iterrows(), total=len(df_final)):
    # Process image
    image = preprocess(Image.open(row['image_path'])).unsqueeze(0).to(device)
    text = clip.tokenize([row['text']], truncate=True).to(device)

    with torch.no_grad():
        image_feat = model.encode_image(image)
        text_feat = model.encode_text(text)

    image_features_list.append(image_feat.cpu().numpy().flatten())
    text_features_list.append(text_feat.cpu().numpy().flatten())

import numpy as np
image_features = np.array(image_features_list)
text_features = np.array(text_features_list)

print("Image features shape:", image_features.shape)
print("Text features shape:", text_features.shape)

100%|██████████| 1895/1895 [00:58<00:00, 32.24it/s]

Image features shape: (1895, 512)
Text features shape: (1895, 512)


In [12]:
import numpy as np

# Combine image + text features into one representation per sample
combined_features = np.concatenate([image_features, text_features], axis=1)
print("Combined features shape:", combined_features.shape)

labels = df_final['label'].values

Combined features shape: (1895, 1024)


In [13]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

X_train, X_test, y_train, y_test = train_test_split(
    combined_features, labels, test_size=0.2, random_state=42, stratify=labels
)

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=['Real', 'Fake']))

Accuracy: 0.8601583113456465
              precision    recall  f1-score   support

        Real       0.89      0.88      0.89       232
        Fake       0.81      0.83      0.82       147

    accuracy                           0.86       379
   macro avg       0.85      0.85      0.85       379
weighted avg       0.86      0.86      0.86       379



In [15]:
def predict_multimodal(image_path, text):
    image = preprocess(Image.open(image_path)).unsqueeze(0).to(device)
    tokenized_text = clip.tokenize([text], truncate=True).to(device)

    with torch.no_grad():
        img_feat = model.encode_image(image).cpu().numpy().flatten()
        txt_feat = model.encode_text(tokenized_text).cpu().numpy().flatten()

    combined = np.concatenate([img_feat, txt_feat]).reshape(1, -1)
    prediction = clf.predict(combined)[0]
    probability = clf.predict_proba(combined)[0]

    label = "FAKE" if prediction == 1 else "REAL"
    confidence = probability[prediction] * 100

    print(f"Prediction: {label}")
    print(f"Confidence: {confidence:.2f}%")

In [16]:
predict_multimodal('images/0.jpg', df_final.iloc[0]['text'])
print("Actual label:", "FAKE" if df_final.iloc[0]['label'] == 1 else "REAL")

Prediction: REAL
Confidence: 99.99%
Actual label: REAL


In [17]:
from google.colab import files
uploaded = files.upload()

Saving IMG-20170123-WA0004.jpg to IMG-20170123-WA0004.jpg


In [18]:
predict_multimodal('IMG-20170123-WA0004.jpg', 'A photo of my lunch today')

Prediction: REAL
Confidence: 99.34%


In [19]:
predict_multimodal('IMG-20170123-WA0004.jpg', 'SHOCKING: scientists baffled by this unexplainable discovery')

Prediction: REAL
Confidence: 99.89%
